## 1. Lectura correcta de los CSV

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


def clean_col_name(col: str) -> str:
    col = col.strip()
    col = col.replace("µ", "u")
    col = col.replace("²", "2")
    col = col.replace("⁶", "6")
    col = col.replace(":", "_")
    col = col.replace("-", "_")
    col = col.replace("/", "_")
    col = col.replace(",", "_")
    col = col.replace("(", "")
    col = col.replace(")", "")
    col = col.replace("%", "pct")
    col = re.sub(r"\s+", "_", col)
    col = re.sub(r"[^a-zA-Z0-9_]", "", col)
    col = re.sub(r"_+", "_", col)
    return col.lower().strip("_")


def load_visem_csv(filename: str, prefix: str | None = None) -> pd.DataFrame:
    path = RAW_DIR / filename

    df = pd.read_csv(
        path,
        sep=";",
        decimal=",",
        encoding="utf-8"
    )

    df.columns = [clean_col_name(c) for c in df.columns]

    if "id" not in df.columns:
        raise ValueError(f"{filename} no tiene columna ID después de limpiar nombres.")

    df["id"] = df["id"].astype(int)

    if prefix is not None:
        rename_map = {
            col: f"{prefix}_{col}"
            for col in df.columns
            if col != "id"
        }
        df = df.rename(columns=rename_map)

    return df

## 2. Cargar cada CSV con prefijos

In [2]:
participant_df = load_visem_csv(
    "participant_related_data.csv",
    prefix="participant"
)

serum_df = load_visem_csv(
    "fatty_acids_serum.csv",
    prefix="serum"
)

sperm_fa_df = load_visem_csv(
    "fatty_acids_spermatoza.csv",
    prefix="sperm_fa"
)

hormones_df = load_visem_csv(
    "sex_hormones.csv",
    prefix="hormone"
)

semen_df = load_visem_csv(
    "semen_analysis_data.csv",
    prefix="semen"
)

videos_df = load_visem_csv(
    "videos.csv",
    prefix=None
)

videos_df = videos_df.rename(columns={"video": "video_filename"})

# Verficar las dimensiones de cada DataFrame
for name, df in {
    "participant": participant_df,
    "serum": serum_df,
    "sperm_fa": sperm_fa_df,
    "hormones": hormones_df,
    "semen": semen_df,
    "videos": videos_df,
}.items():
    print(f"{name} - Shape: {df.shape}".center(80, "="))
    print(df.head(2))
    print()

==========================participant - Shape: (85, 4)==========================
   id participant_abstinence_timedays  participant_body_mass_index_kg_m2  \
0   1                             4,0                               32.5   
1   2                             4,0                               33.7   

   participant_age_years  
0                     36  
1                     61  

============================serum - Shape: (85, 16)=============================
   id  serum_serum_c14_0_myristic_acid  serum_serum_c16_0_palmitic_acid  \
0   1                             0.36                            29.72   
1   2                             0.28                            31.22   

   serum_serum_c16_1_palmitoleic_acid  serum_serum_c18_0_stearic_acid  \
0                                0.64                           13.67   
1                                0.47                           11.84   

   serum_serum_c18_1_n_9_oleic_acid  serum_serum_total_c18_1  \
0                

## 3. Crear tabla clínica sin semen analysis

In [3]:
# Esta tabla sera la X_Clinical
clinical_features = (
    participant_df
    .merge(serum_df, on="id", how="inner")
    .merge(sperm_fa_df, on="id", how="inner")
    .merge(hormones_df, on="id", how="inner")
    .merge(videos_df, on="id", how="left")
)

print(clinical_features.shape)
clinical_features.head()

# Guardar la tabla clínica sin semen analysis
clinical_features.to_csv(
    PROCESSED_DIR / "clinical_features.csv",
    index=False
)

(85, 51)


## 4. Crear targets proxy desde semen analysis
### Targets individuales

In [4]:
targets = semen_df[["id"]].copy()

targets["target_low_concentration"] = (
    semen_df["semen_sperm_concentration_x106_ml"] < 16
).astype(int)

targets["target_low_total_sperm_count"] = (
    semen_df["semen_total_sperm_count_x106"] < 39
).astype(int)

targets["target_low_progressive_motility"] = (
    semen_df["semen_progressive_motility_pct"] < 30
).astype(int)

targets["target_low_total_motility"] = (
    (
        semen_df["semen_progressive_motility_pct"]
        + semen_df["semen_non_progressive_sperm_motility_pct"]
    ) < 42
).astype(int)

targets["target_low_vitality"] = (
    semen_df["semen_sperm_vitality_pct"] < 54
).astype(int)

targets["target_low_morphology"] = (
    semen_df["semen_normal_spermatozoa_pct"] < 4
).astype(int)

### Target general
Este target indica si el participante tiene al menos una alteración seminal según esas reglas

In [5]:
target_cols = [
    "target_low_concentration",
    "target_low_total_sperm_count",
    "target_low_progressive_motility",
    "target_low_total_motility",
    "target_low_vitality",
    "target_low_morphology",
]

targets["target_any_semen_abnormality"] = (
    targets[target_cols].sum(axis=1) > 0
).astype(int)

targets.head()

,id,target_low_concentration,target_low_total_sperm_count,target_low_progressive_motility,target_low_total_motility,target_low_vitality,target_low_morphology,target_any_semen_abnormality
0,1,0,0,0,0,0,1,1
1,2,0,0,1,1,1,1,1
2,3,0,0,1,0,0,0,1
3,4,0,0,0,0,0,1,1
4,5,0,0,0,0,0,1,1


Guardar los targets:

In [6]:
targets.to_csv(
    PROCESSED_DIR / "semen_targets.csv",
    index=False
)

## 5. Validación final

In [7]:
import pandas as pd

clinical_features = pd.read_csv("../data/processed/clinical_features.csv")
semen_targets = pd.read_csv("../data/processed/semen_targets.csv")

print("clinical_features:", clinical_features.shape)
print("semen_targets:", semen_targets.shape)

print("\nIDs únicos clinical:")
print(clinical_features["id"].nunique())

print("\nIDs únicos targets:")
print(semen_targets["id"].nunique())

print("\nDuplicados clinical:")
print(clinical_features["id"].duplicated().sum())

print("\nDuplicados targets:")
print(semen_targets["id"].duplicated().sum())

print("\nDistribución del target general:")
print(semen_targets["target_any_semen_abnormality"].value_counts())
print(semen_targets["target_any_semen_abnormality"].value_counts(normalize=True))

print("\nMissing values clinical:")
display(
    clinical_features
    .isna()
    .mean()
    .sort_values(ascending=False)
    .head(30)
)

print("\nMissing values targets:")
display(
    semen_targets
    .isna()
    .mean()
    .sort_values(ascending=False)
)

clinical_features: (85, 51)
semen_targets: (85, 8)

IDs únicos clinical:
85

IDs únicos targets:
85

Duplicados clinical:
0

Duplicados targets:
0

Distribución del target general:
target_any_semen_abnormality
1    59
0    26
Name: count, dtype: int64
target_any_semen_abnormality
1    0.694118
0    0.305882
Name: proportion, dtype: float64

Missing values clinical:


id                                                 0.0
participant_abstinence_timedays                    0.0
participant_body_mass_index_kg_m2                  0.0
participant_age_years                              0.0
serum_serum_c14_0_myristic_acid                    0.0
serum_serum_c16_0_palmitic_acid                    0.0
serum_serum_c16_1_palmitoleic_acid                 0.0
serum_serum_c18_0_stearic_acid                     0.0
serum_serum_c18_1_n_9_oleic_acid                   0.0
serum_serum_total_c18_1                            0.0
serum_serum_c18_2_n_6_linoleic_acid_la             0.0
serum_serum_c18_3_n_6_gamma_linoleic_acid_gla      0.0
serum_serum_c20_1_n_9                              0.0
serum_serum_c20_2_n_6                              0.0
serum_serum_c20_3_n_6                              0.0
serum_serum_c20_4_n_6                              0.0
serum_serum_c20_5_n_3_eicosapentaenoic_acid_epa    0.0
serum_serum_c22_5_n_3_docosapentaenoic_acid_dpa    0.0
serum_seru


Missing values targets:


id                                 0.0
target_low_concentration           0.0
target_low_total_sperm_count       0.0
target_low_progressive_motility    0.0
target_low_total_motility          0.0
target_low_vitality                0.0
target_low_morphology              0.0
target_any_semen_abnormality       0.0
dtype: float64